# 1. import library

In [2]:
import os
import glob
from tqdm import tqdm

import numpy as np
import pandas as pd
from scipy.optimize import curve_fit

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.font_manager as fm
import matplotlib.colors as mcolors
import matplotlib as mpl
from matplotlib.colors import PowerNorm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from svgutils.compose import Figure, SVG

# 2. Load data

In [ ]:
def load_csv(genealogy_file_list):
    all_genealogy_data = []

    for file in genealogy_file_list:
        df = pd.read_csv(file)

        # add some col
        file_name = os.path.basename(file)
        df['file_name'] = file_name

        if 'PY1' in file:
            df['Specie'] = 'PY1'
        elif 'europaea' in file:
            df['Specie'] = 'europaea'
        else:
            df['Specie'] = 'Unknown'

        df['Time'] = df['Time'].round(2)

        all_genealogy_data.append(df)

    All_genealogy_data = pd.concat(all_genealogy_data, ignore_index=True)
    
    All_genealogy_data_2 = All_genealogy_data.copy()
    All_genealogy_data_2['replicate'] = All_genealogy_data_2['file_name'].apply(
        lambda x: 1 if 'N1_' in x else 2 if 'N2_' in x else 3 if 'N3_' in x else 0
    )
    All_genealogy_data_2['FOV'] = All_genealogy_data_2['file_name'].apply(
        lambda x: 1 if '_1' in x else 2 if '_2' in x else 3 if '_3' in x else 0
    )
    All_genealogy_data_2['Condition'] = All_genealogy_data_2['file_name'].apply(
        lambda x: 'no-supernatant' if '_no-supernatant' in x
        else 'supernatant' if '_supernatant' in x
        else ''
    )

    cols = All_genealogy_data_2.columns.tolist()
    cols_reordered = ['Specie', 'Condition', 'replicate', 'FOV'] + [col for col in cols if col not in ['Specie', 'Condition', 'replicate', 'FOV', 'file_name']]
    All_genealogy_data_2 = All_genealogy_data_2[cols_reordered]
    
    return(All_genealogy_data_2)


In [ ]:
def mutate_genealogy_data(All_genealogy_data):
    # Step 1: make spot_id_set in each group
    spot_id_set_df = (
        All_genealogy_data
        .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'track', 'generation_count'], as_index=False)
        .agg(spot_id_set=('Spot.ID', lambda x: sorted(x.tolist())))
        .drop_duplicates(subset=['spot_id_set'])
    )

    # Step 2: inner join with original data
    All_genealogy_data_2 = pd.merge(
        spot_id_set_df.drop(columns=['spot_id_set']),
        All_genealogy_data,
        on=['Specie', 'Condition', 'replicate', 'FOV', 'track', 'generation_count'],
        how='inner'
    )

    # Step 3: move generation_count col and track col to latter col.
    cols = All_genealogy_data_2.columns.tolist()
    cols.remove('generation_count')
    cols.remove('track')
    All_genealogy_data_2 = All_genealogy_data_2[cols + ['generation_count', 'track']]

    # Step 4: make IF_Divide col in each geneanology base on IF_Divided col
    All_genealogy_data_2.sort_values(['Specie', 'Condition', 'replicate', 'FOV', 'track', 'Time'], inplace=True)
    All_genealogy_data_2['IF_Divide'] = (
        All_genealogy_data_2
        .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'track'])['IF_Divided']
        .shift(-1)
    )

    # Step 5: make IF_will_Divide col
    def determine_will_divide(group):
        if 'Y' in group['IF_Divide'].values:
            group['IF_will_Divide'] = 'divide'
        else:
            group['IF_will_Divide'] = 'non-divide'
        return group

    All_genealogy_data_2 = (
        All_genealogy_data_2
        .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'track', 'generation_count'], group_keys=False)
        .apply(determine_will_divide)
        .reset_index(drop=True)
    )

    # Step 6: calculate Time_start, Time_end, generation_time
    time_stats = (
        All_genealogy_data_2
        .groupby(['Specie', 'Condition', 'replicate', 'FOV', 'track', 'generation_count'])['Time']
        .agg(Time_start='min', Time_end='max')
        .reset_index()
    )
    time_stats['generation_time'] = time_stats['Time_end'] - time_stats['Time_start']

    # Step 7: concatenate
    All_genealogy_data_2 = pd.merge(
        All_genealogy_data_2,
        time_stats,
        on=['Specie', 'Condition', 'replicate', 'FOV', 'track', 'generation_count'],
        how='left'
    )
    
    # Step 8: additonal
    def compute_growth_features(df):
        df = df.sort_values("Frame")
        Ab = df["Area"].iloc[0]
        Ad = df["Area"].iloc[-1]
        Tb = df["Time"].iloc[0]
        Td = df["Time"].iloc[-1]

        df["Growth_rate_btw_birth_div"] = np.log(Ad / Ab) / (Td - Tb) if Ab > 0 and (Td - Tb) > 0 else np.nan

        return df

    All_genealogy_data_3 = All_genealogy_data_2.groupby(
        ["Specie", 'Condition', 'replicate', 'FOV', "track", "generation_count"],
        group_keys=False
    ).apply(compute_growth_features).reset_index(drop=True)
    All_genealogy_data_3['age'] = All_genealogy_data_3['Time'] - All_genealogy_data_3['Time_start']
    
    return(All_genealogy_data_3)


In [ ]:
genealogy_file_list = glob.glob("../0_rawdata/genealogy_data_PY1/*.csv") + glob.glob("../0_rawdata/genealogy_data_europaea/*.csv")
All_genealogy_data_raw = load_csv(genealogy_file_list)

All_genealogy_data_mutate = mutate_genealogy_data(All_genealogy_data_raw)

# 3. fit to exponential model

In [ ]:
def elongation_exp_fit(cells):
    def exp_growth(t, A0, k):
        return A0 * np.exp(k * t)

    fit_results = []
    for (specie, cond, rep, fov, track, Gcount), group in tqdm(
        cells.groupby(["Specie", "Condition", "replicate", "FOV", "track", "generation_count"])
        ):
        group = group.sort_values("Time")
        t = group["age"].values
        Tb = group["Time_start"].iloc[0]
        Td = group["Time_end"].iloc[0]
        generation_time = group["generation_time"].iloc[0]
        A = group["Area"].values
        m = group["Growth_rate_btw_birth_div"].unique()

        # Exclude genealogy groups with fewer than four data points
        if len(t) < 4:
            continue

        try:
            popt, pcov = curve_fit(exp_growth, t, A, p0=[A[0], m[0]])
            A0, k = popt
            residuals = A - exp_growth(t, *popt)
            ss_res = np.sum(residuals**2)
            ss_tot = np.sum((A - np.mean(A))**2)
            r_squared = 1 - (ss_res / ss_tot)

            fit_results.append({
                "Specie": specie,
                "Condition": cond,
                "replicate": rep,
                "FOV": fov,
                "track": track,
                "generation_count": Gcount,
                "Tb": Tb,
                "Td": Td,
                "generation_time": generation_time,
                "A0": A0,
                "k": k,
                "r_squared": r_squared,
                "n_points": len(t)
            })

        except RuntimeError:
            continue

    fit_df = pd.DataFrame(fit_results)
    
    return(fit_df)

In [ ]:
# Select genealogies that undergo cell division
dividing_cells = All_genealogy_data_mutate[All_genealogy_data_mutate["IF_will_Divide"] == "divide"]

# Remove entries with missing or non-finite area or time values
dividing_cells_clean = dividing_cells[
	dividing_cells["Area"].notna() &
	dividing_cells["Time"].notna() &
	np.isfinite(dividing_cells["Area"]) &
	np.isfinite(dividing_cells["Time"])
]

fit_df = elongation_exp_fit(dividing_cells_clean)

# 4. Plot

## 4.1. config

In [ ]:
def set_mytheme_paper(ax):
    # font
    plt.rcParams["text.usetex"] = False
    plt.rcParams["font.family"] = "Helvetica"
    plt.rcParams["font.size"] = 8
    plt.rcParams["text.color"] = "black"
    mpl.rcParams['svg.fonttype'] = 'none'

    # title
    ax.title.set_fontsize(9.5)
    ax.title.set_color("black")
    ax.title.set_position((0.5, 1.05))

    # axis title
    ax.xaxis.label.set_size(8)
    ax.yaxis.label.set_size(8)
    ax.xaxis.label.set_color("black")
    ax.yaxis.label.set_color("black")

    # ticks
    ax.tick_params(axis='x', labelsize=6.5, colors="black")
    ax.tick_params(axis='y', labelsize=6.5, colors="black")

    # spine
    for spine in ax.spines.values():
        spine.set_color("black")
        spine.set_linewidth(1.0)

    # figure bacground
    ax.set_facecolor("none")
    ax.figure.set_facecolor("none")

    # grid
    ax.grid(False)

    # legend
    ax.legend(
        loc="upper left",
        frameon=False,
        fontsize=6.5
    )
    

In [4]:
folder_path = "./result(plots)"
os.makedirs(folder_path, exist_ok=True)

## 4.2. function

In [ ]:
def plot_fitting_result(fit_df, All_genealogy_data_mutate, folder_path=None, 
                        file_name=None, base_name=None, IF_show=False, panel_label=False):
    def exp_growth(t, A0, k):
        return A0 * np.exp(k * t)
    
    # --- label ---
    specie_colors = {
        "PY1": "salmon",
        "europaea": "royalblue"
    }

    # --- plot ---
    n_cols, n_rows = 3, 1
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.4*n_cols, 2.4*n_rows))
    axes = axes.flatten()

    for ax, (_, row) in zip(axes, fit_df.iterrows()):
        specie = row["Specie"]
        condition = row["Condition"]
        rep, fov, track = row["replicate"], row["FOV"], row["track"]

        group = All_genealogy_data_mutate[
            (All_genealogy_data_mutate["Specie"] == specie) &
            (All_genealogy_data_mutate["Condition"] == condition) &
            (All_genealogy_data_mutate["replicate"] == rep) &
            (All_genealogy_data_mutate["FOV"] == fov) &
            (All_genealogy_data_mutate["track"] == track)
        ]

        t = group["Time"].values
        A = group["Area"].values
        age = group["age"].values
        A_fit = exp_growth(age, row["A0"], row["k"])

        ax.plot(t, A, 'o', label="Observed",
                markersize=3, color=specie_colors[specie])
        ax.plot(t, A_fit, '-', label="Fitted", 
                linewidth = 2, color='black')
        ax.set_title(fr"R$^2$={row['r_squared']:.3f}")
        ax.set_xlabel("Time (h)")
        ax.set_ylabel(r"Cell area ($\mathrm{\mu m^2}$)")
        set_mytheme_paper(ax)
        
    # --- add panel label ---
    if panel_label is not None:
        fig.text(
            0.01, 0.99, panel_label,
            fontsize=12, fontweight='bold',
            va='top', ha='left',
            color='black'
        )

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    # --- display ---
    if IF_show:
        plt.show()
    else:
        plt.close()
    
    # --- save ---
    if folder_path and base_name:
        output_path = os.path.join(folder_path, base_name)
        os.makedirs(output_path, exist_ok=True)
        
        svg_file = f"{file_name}.svg"
        png_file = f"{file_name}.png"
        fig.savefig(os.path.join(output_path, svg_file), format="svg")
        fig.savefig(os.path.join(output_path, png_file), format="png", dpi=600)

In [ ]:
def plot_fitting_result_representative(fit_row, All_genealogy_data_mutate,
                                       folder_path=None, base_name=None, 
                                       IF_show = False, axis_limits=None):
    def exp_growth(t, A0, k):
        return A0 * np.exp(k * t)
    
    # --- label ---
    specie_colors = {
        "PY1": "salmon",
        "europaea": "royalblue"
    }
    specie_labels = {
        "PY1": "Nitrosomonas sp. PY1",
        "europaea": "N. europaea"
    }
    specie_tags = {
        "PY1_no-supernatant": "B",
        "europaea_no-supernatant": "S1",
        "PY1_supernatant": "S3"
    }
    
    # --- data selection ---
    specie = fit_row["Specie"]
    condition = fit_row["Condition"]
    rep = fit_row["replicate"]
    fov = fit_row["FOV"]
    track = fit_row["track"]

    group = All_genealogy_data_mutate[
        (All_genealogy_data_mutate["Specie"] == specie) &
        (All_genealogy_data_mutate["Condition"] == condition) &
        (All_genealogy_data_mutate["replicate"] == rep) &
        (All_genealogy_data_mutate["FOV"] == fov) &
        (All_genealogy_data_mutate["track"] == track)
    ]

    t = group["Time"].values
    A = group["Area"].values
    age = group["age"].values
    A_fit = exp_growth(age, fit_row["A0"], fit_row["k"])

    # --- plot ---
    fig, ax = plt.subplots(figsize=(2.4, 2.4))
    ax.plot(t, A, 'o', 
            label="Experimental data", color=specie_colors[specie], markersize=3)
    ax.plot(t, A_fit, 
            '-', label="Exponential fit", color='black', linewidth=2)
    ax.set_xlabel("Time (h)")
    ax.set_ylabel(r"Cell area ($\mathrm{\mu m^2}$)")
    fig.text(0.05, 0.95, 
             specie_tags[specie+"_"+condition], fontweight='bold', fontsize = 12)
    
    # --- set axis ---
    if axis_limits is not None:
        ax.set_xlim(axis_limits["xlim"])
        ax.set_ylim(axis_limits["ylim"])

    plt.tight_layout()
    set_mytheme_paper(ax) 

    # --- display ---
    if IF_show:
        plt.show()
    else:
        plt.close()
    
    # --- save ---
    if folder_path:
        folder_path_2 = os.path.join(folder_path, "representative_cell_trajectory")
        os.makedirs(folder_path_2, exist_ok=True)
        
        svg_file = f"{folder_path_2}/{base_name}.svg"
        png_file = f"{folder_path_2}/{base_name}.png"
        fig.savefig(svg_file, format="svg")
        fig.savefig(png_file, format="png", dpi=600)
        

In [ ]:
def calculate_axis_limits(fit_data):
    x_min = fit_data["Tb"].min() - fit_data["generation_time"].max()/20
    x_max = fit_data["Td"].max() + fit_data["generation_time"].max()/20
    y_min = fit_data["A0"].min() * 0.8
    y_max = max([
        row["A0"] * np.exp(row["k"] * row["generation_time"])
        for _, row in fit_data.iterrows()
    ]) * 1.05
    return {"xlim": (x_min, x_max), "ylim": (y_min, y_max)}

In [ ]:
def plot_multiple_growth_trajectories(fit_data,
                                      folder_path=None, base_name=None, 
                                      IF_show=False, axis_limits=None, 
                                      n_bins=20, n_per_bin=20, random_state=1):    
    def exp_growth(t, A0, k):
        return A0 * np.exp(k * t)
    
    # --- label ---
    specie_colors = {
        "PY1": "salmon",
        "europaea": "royalblue"
    }
    specie_tags = {
        "PY1_no-supernatant": "C",
        "europaea_no-supernatant": "D",
        "PY1_supernatant": "S3"
    }
    
    # --- data selection ---
    specie = fit_data["Specie"].iloc[0]
    Cond = fit_data['Condition'].iloc[0]
    Rep = fit_data['replicate'].iloc[0]

    # --- random sampling ---
    background_rows = []
    rng = np.random.default_rng(seed=random_state)
    bins = np.linspace(fit_data["Tb"].min(),
                       fit_data["Tb"].max(),
                       n_bins+1)
    for i in range(n_bins):
        bin_data = fit_data[(fit_data["Tb"] >= bins[i]) & 
                            (fit_data["Tb"] < bins[i+1])]
        if len(bin_data) > 0:
            background_rows.append(bin_data
                                   .sample(min(n_per_bin, len(bin_data)),
                                           random_state=rng.integers(1e9)))
    background_data = pd.concat(background_rows)

    # --- plot ---
    fig, ax = plt.subplots(figsize=(3.2, 2.4))
    
    # color
    base_color = mcolors.to_rgb("gray")
    mid_color  = mcolors.to_rgb("gray") 
    target_color = mcolors.to_rgb(specie_colors[specie])
    cmap = mcolors.LinearSegmentedColormap.from_list(
        f"{specie}_grad", [base_color, mid_color, target_color]
    )
    norm = PowerNorm(gamma=0.3, 
                     vmin=fit_data["generation_time"].min(), 
                     vmax=fit_data["generation_time"].max())
    
    for _, row in background_data.iterrows():
        age = np.linspace(0, row["generation_time"], 50)
        A_fit = exp_growth(age, row["A0"], row["k"])
        time = np.linspace(row["Tb"], row["Td"], 50)
        color = cmap(norm(row["generation_time"]))
        ax.plot(time, A_fit,
                color = color,
                linewidth=1, alpha=0.5, zorder=1)

    # axis
    if axis_limits is not None:
        x_min = fit_data["Tb"].min() - fit_data["generation_time"].max()/20
        x_max = fit_data["Td"].max() + fit_data["generation_time"].max()/20
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(axis_limits["ylim"])
    else:
        x_min = fit_data["Tb"].min() * 0.9 
        x_max = fit_data["Td"].max() * 1.1
        y_min = fit_data["A0"].min() * 0.9
        y_max = max([
            row["A0"] * np.exp(row["k"] * row["generation_time"])
            for _, row in fit_data.iterrows()
        ]) * 1.1
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
    ax.set_xlabel("Time (h)")
    ax.set_ylabel(r"Cell area ($\mathrm{\mu m^2}$)")
    fig.text(0.05, 0.95, specie_tags[specie+"_"+Cond], fontweight='bold', fontsize = 12)

    plt.tight_layout()
    set_mytheme_paper(ax) 
    
    # color bar
    sm = cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cax = fig.add_axes([0.225, 0.825, 0.3, 0.03])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.ax.xaxis.set_label_position("top")
    cbar.set_label("Generation time (h)", fontsize=8, fontname="Helvetica")
    font_prop = fm.FontProperties(family="Helvetica", size=6.5)
    for tick in cbar.ax.get_xticklabels():
        tick.set_fontproperties(font_prop)
    
    # ticks
    vmin, vmax = fit_data["generation_time"].min(), fit_data["generation_time"].max()
    ticks = np.linspace(vmin, vmax, 4)
    cbar.set_ticks(ticks)
    cbar.set_ticklabels([str(int(t)) for t in ticks])

    # --- save ---
    if folder_path:
        output_folder = os.path.join(folder_path, base_name)
        os.makedirs(output_folder, exist_ok=True)
        
        svg_file = os.path.join(output_folder, f"{specie}_{Cond}_N{Rep}.svg")
        png_file = os.path.join(output_folder, f"{specie}_{Cond}_N{Rep}.png")
        fig.savefig(svg_file, bbox_inches='tight')
        fig.savefig(png_file, dpi=600, bbox_inches='tight')
    
    # --- display ---
    if IF_show:
        plt.show()
    else:
        plt.close()

## 4.3. data selection

In [ ]:
# data selection
target_Gcount = 1
r_square_thresh = 0.75
exp_data_filtered = dividing_cells[(dividing_cells["generation_count"] == target_Gcount)]
fit_data_filtered = fit_df[(fit_df["generation_count"] == target_Gcount) &
                                 (fit_df["r_squared"] > r_square_thresh) &
                                 (fit_df["k"] > 0)
                                 ]

# exstract top 10 R^2 trajectory
fit_data_top10_list = []
fit_data_sorted = (fit_data_filtered
                   .sort_values(by="r_squared", ascending=False)
                   .groupby(["Specie", "Condition"])
                   )
for (specie, condition), group in fit_data_sorted:
    fit_data_top10_list.append(group.head(10))
fit_data_top10 = pd.concat(fit_data_top10_list)

# extract various R^2 trajectory uniformly
fit_data_various_R_squared_list = []
for (specie, condition), group in fit_data_sorted:
    if len(group) >= 3:
        idxs = np.linspace(0, len(group)-1, 3, dtype=int)
        fit_data_various_R_squared_list.append(group.iloc[idxs])
    else:
        fit_data_various_R_squared_list.append(group)
fit_data_various_R_squared = pd.concat(fit_data_various_R_squared_list)

## 4.4. top 10 R² plot

In [ ]:
for (specie, condition), group in fit_data_top10.groupby(["Specie", "Condition"]):
    base_name = "top10_r^2"
    file_name = f"{specie}_{condition}"
    plot_fitting_result(group, exp_data_filtered, folder_path=folder_path, 
                        file_name=file_name, base_name=base_name, IF_show=False)

## 4.5. representative plot

In [ ]:
target_cells = [
    ("PY1", "no-supernatant", 1, 2, 31),
    ("europaea", "no-supernatant", 2, 1, 398),
    ("PY1", "supernatant", 2, 3, 146)
]

axis_limits_dict = {}
for specie_val, condition_val, N_val, FOV_val, track_val in target_cells:
    fit_df_filtered = fit_data_top10[
        (fit_data_top10["Specie"] == specie_val) &
        (fit_data_top10["Condition"] == condition_val) &
        (fit_data_top10["replicate"] == N_val) &
        (fit_data_top10["FOV"] == FOV_val) &
        (fit_data_top10["track"] == track_val)
    ]
    axis_limits_dict[(specie_val, condition_val, N_val, FOV_val, track_val)] = calculate_axis_limits(fit_df_filtered)

for specie_val, condition_val, N_val, FOV_val, track_val in target_cells:
    group_df = fit_data_top10[
        (fit_data_top10["Specie"] == specie_val) &
        (fit_data_top10["Condition"] == condition_val) &
        (fit_data_top10["replicate"] == N_val) &
        (fit_data_top10["FOV"] == FOV_val) &
        (fit_data_top10["track"] == track_val)
    ]
    fit_row = group_df.iloc[0]
    base_name = f"{specie_val}_{condition_val}_N{N_val}_FOV{FOV_val}_track{track_val}"
    axis_limits = axis_limits_dict[(specie_val, condition_val, N_val, FOV_val, track_val)]
    
    # axis_limits = axis_limits_dict[("PY1", "no-supernatant", 1, 2, 31)]
    plot_fitting_result_representative(fit_row, exp_data_filtered,
                                       folder_path=folder_path, base_name=base_name, 
                                       IF_show=False, axis_limits=axis_limits)


## 4.6. Uniformly sampled R² (0.75–1.0) plot

In [ ]:
panel_map = {
    ("PY1", "no-supernatant"): "A",
    ("europaea", "no-supernatant"): "B",
    ("PY1", "supernatant"): "C",
}

for (specie, condition), group in fit_data_various_R_squared.groupby(["Specie", "Condition"]):
    base_name = "various_r^2"
    file_name = f"{specie}_{condition}"
    panel_label = panel_map.get((specie, condition), "")
    plot_fitting_result(group, exp_data_filtered, folder_path=folder_path,
                        base_name=base_name, file_name = file_name, IF_show=False, panel_label=panel_label)

## 4.7. Overlay of multiple growth trajectories

In [ ]:
target_N = [
    ("PY1", "no-supernatant", 2),
    ("europaea", "no-supernatant", 2),
    ("PY1", "supernatant", 2)
]

r_square_thresh = 0.75
axis_limits_dict = {}
for specie_val, condition_val, N_val in target_N:
    fit_df_filtered = fit_df[
        (fit_df["Specie"] == specie_val) &
        (fit_df["Condition"] == condition_val) &
        (fit_df["replicate"] == N_val) &
        (fit_df["generation_count"] != 0) &
        (fit_df["r_squared"] > r_square_thresh) &
        (fit_df["k"] > 0)
    ]
    axis_limits_dict[(specie_val, condition_val, N_val)] = calculate_axis_limits(fit_df_filtered)

for specie_val, condition_val, N_val in target_N:
    fit_df_filtered = fit_df[
        (fit_df["Specie"] == specie_val) &
        (fit_df["Condition"] == condition_val) &
        (fit_df["replicate"] == N_val) &
        (fit_df["generation_count"] != 0) &
        (fit_df["r_squared"] > r_square_thresh) &
        (fit_df["k"] > 0)
        ]
    
    base_name = f"fig1C_overlay_of_multiple_growth_trajectory"
    axis_limits = axis_limits_dict["PY1", "no-supernatant", 2]
    plot_multiple_growth_trajectories(fit_df_filtered, 
                                      folder_path=folder_path, base_name=base_name,
                                      IF_show=False, axis_limits=axis_limits)

# 5. export

In [ ]:
def export_df(fit_df, r_square_thr, All_genealogy_data):
    # --- check used data amount ---
    conditions_to_check = [
        ("PY1", "no-supernatant"),
        ("PY1", "supernatant"),
        ("europaea", "no-supernatant"),
        ("europaea", "supernatant")
    ]
    for specie, condition in conditions_to_check:
        total_count = len(fit_df[(fit_df["Specie"]==specie) &
                                 (fit_df["Condition"]==condition)])
        if total_count==0:
            continue
        subset = fit_df[
            (fit_df["Specie"]==specie) &
            (fit_df["Condition"]==condition) &
            (fit_df["r_squared"] > r_square_thr) &
            (fit_df["k"] > 0)
        ]
        used_count = len(subset)
        
        print(
            f"Specie={specie}, Condition={condition} -> "
            f"Total: {total_count}, Used: {used_count} "
            f"({used_count/total_count*100:.1f}%)")
        
    # --- export fitted data ---
    def exp_growth(t, A0, k):
        return A0 * np.exp(k * t)
    annotated_dfs = []
    
    grouped_dict = {
        key: group
        for key, group in (All_genealogy_data
                           .groupby(["Specie", "Condition", "replicate", "FOV", "track", "generation_count"])
                           )
    }
    fit_df_filtered = fit_df[(fit_df["r_squared"] > r_square_thr) &
                             (fit_df["k"] > 0)]
        
    for _, row in fit_df_filtered.iterrows():
        key = ((row["Specie"], row["Condition"], row["replicate"], row["FOV"], row["track"], row["generation_count"]))
        group = grouped_dict.get(key)
        if group is None:
            continue
        
        age = group['age'].values
        A_fit = exp_growth(age, row["A0"], row["k"])
        
        group = group.copy()
        group["fitted_area"] = A_fit
        group["fitted_elongation_rate"] = row["k"]
        annotated_dfs.append(group)

    combined_df = pd.concat(annotated_dfs, ignore_index=True)    
    return(combined_df)

In [ ]:
combined_df_exponential = export_df(fit_df, r_square_thresh, dividing_cells_clean)
combined_df_exponential.to_csv("./processed_data/2_fitted_genealogy_data_exponential.csv", index=False)

In [5]:
prefix = os.path.join(folder_path, "various_r^2")
pt_to_mm = 25.4 / 72

Figure(
    "183mm", "183mm",
    SVG(os.path.join(prefix, "PY1_no-supernatant.svg")).scale(pt_to_mm).move(0, 0),
    SVG(os.path.join(prefix, "europaea_no-supernatant.svg")).scale(pt_to_mm).move(0, 2.4*25.4),
    SVG(os.path.join(prefix, "PY1_supernatant.svg")).scale(pt_to_mm).move(0, 2.4*25.4*2),
).save(os.path.join(prefix, "fig.S7.svg"))